# External Validation Experiment

## Base Paper
**Explainable Machine Learning Techniques To Predict  
Amiodarone-Induced Thyroid Dysfunction Risk:  
Multicenter, Retrospective Study With External Validation**

## Description
This code performs external validation based on the methodology and model
described in the base paper above, using an independent dataset.

## Objective
To assess the generalizability and predictive performance of the explainable
machine learning model on an external cohort from Gil Medical Center, Korea.

## Dataset
- **Gil Medical Center (Korea)**

## Experimenter
- **WonJong Yu**
- Gachon University

In [ ]:
import pandas as pd
import numpy as np
from pandas.tseries.offsets import DateOffset
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import RobustScaler

# Preprocessing

In [ ]:
RANDOM_STATE = 42

## Data loading

In [ ]:
df = pd.read_csv("dataset.csv")

In [ ]:
df.head()

In [ ]:
for col in df.columns:
    print(col)

In [ ]:
# Outlier detection in height and weight

df.loc[
    (df['height_cm'] < 120) | (df['height_cm'] > 220) |
    (df['weight_kg'] < 30) | (df['weight_kg'] > 250),
    ['person_id', 'height_cm', 'weight_kg', 'bmi', 'aitd_flag']
].head(100)

In [ ]:
# Treat outliers in height and weight as missing values

invalid_height = (df['height_cm'] < 120) | (df['height_cm'] > 220)
invalid_weight = (df['weight_kg'] < 30) | (df['weight_kg'] > 250)
df.loc[invalid_height, 'height_cm'] = np.nan
df.loc[invalid_weight, 'weight_kg'] = np.nan
df.loc[invalid_height | invalid_weight, 'bmi'] = np.nan

In [ ]:
# Validate the SQL-Python data contract and convert date columns.
required_columns = [
    "person_id", "aitd_flag", "index_date", "analysis_end_date",
    "last_complete_observation_window",
]
missing_required = sorted(set(required_columns) - set(df.columns))
if missing_required:
    raise KeyError(f"dataset.csv is missing required SQL columns: {missing_required}")

date_columns = [
    "index_date", "followup_end_date", "aitd_event_date",
    "feature_cutoff_date", "analysis_end_date",
]
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

In [ ]:
# Use the last complete observation window determined in SQL.
# SQL and Python use the same canonical observation-window labels.
valid_timepoints = {
    "base", "m1", "m2", "m3", "m4_6", "m7_9",
    "m10_12", "m13_15", "m16_18", "m19_21",
}
df["lab_feature_cutoff"] = df["last_complete_observation_window"].where(
    df["last_complete_observation_window"].isin(valid_timepoints)
)
if df["lab_feature_cutoff"].isna().any():
    bad = sorted(df.loc[df["lab_feature_cutoff"].isna(), "last_complete_observation_window"].astype(str).unique())
    raise ValueError(f"Unknown last_complete_observation_window values: {bad}")

In [ ]:
# Check event windows and the feature cutoff selected before each event/censoring point.
pd.crosstab(
    df.get("aitd_event_window", pd.Series(index=df.index, dtype="object")),
    df["lab_feature_cutoff"], dropna=False,
)

In [ ]:
# Time point definitions

time_points = [
    "base", "m1", "m2", "m3", "m4_6",
    "m7_9", "m10_12", "m13_15", "m16_18", "m19_21",
]
tp_index = {tp: i for i, tp in enumerate(time_points)}

labs = [
    "tsh", "ft4", "rbc", "hgb", "hct", "mcv", "mch", "mchc", "tg",
    "ast", "alt", "hdl", "ldl", "scr",
    "alkp", "cholesterol"
]


In [ ]:
# Select the latest actually observed value by moving forward only through
# the SQL-defined last complete observation window.

def select_time_point(row, lab):
    lab_cols = [f"{lab}_{tp}" for tp in time_points]
    values = row[lab_cols]

    cutoff = row["lab_feature_cutoff"]
    if pd.isna(cutoff) or cutoff not in tp_index:
        return pd.NA

    selected = pd.NA
    for i in range(tp_index[cutoff] + 1):
        if pd.notna(values.iloc[i]):
            selected = time_points[i]
    return selected


def select_lab_value(row, lab):
    sel_tp = row[f"{lab}_time_point"]

    if pd.isna(sel_tp):
        return pd.NA

    return row[f"{lab}_{sel_tp}"]


def select_lab_rem(row, lab):
    sel_tp = row[f"{lab}_time_point"]

    if pd.isna(sel_tp):
        return pd.NA

    # SQL uses chol_rem_* for cholesterol remarks.
    rem_prefix = 'chol' if lab == 'cholesterol' else lab
    rem_col = f"{rem_prefix}_rem_{sel_tp}"
    return row[rem_col] if rem_col in row.index else pd.NA

for lab in labs:
    df[f"{lab}_time_point"] = df.apply(
        lambda row: select_time_point(row, lab), axis=1
    )

    df[f"{lab}_value"] = df.apply(
        lambda row: select_lab_value(row, lab), axis=1
    )

    df[f"{lab}_rem"] = df.apply(
        lambda row: select_lab_rem(row, lab), axis=1
    )


In [ ]:
# Store the selected laboratory time point for audit. Slopes are calculated
# in SQL from actual measurement dates, not from these window labels.
timepoint_cols = [f"{lab}_time_point" for lab in labs]
df_timepoints = df.loc[
    :, ["person_id", "last_complete_observation_window", "lab_feature_cutoff", *timepoint_cols]
].copy()

In [ ]:
# Preserve correct time-to-event/censoring data for survival analysis, then
# remove all outcome/censoring metadata from the predictor matrix.
survival_df = df.loc[:, ["person_id", "index_date", "analysis_end_date", "aitd_flag"]].copy()
survival_df["followup_time_days"] = (
    survival_df["analysis_end_date"] - survival_df["index_date"]
).dt.days
if survival_df["followup_time_days"].isna().any() or (survival_df["followup_time_days"] < 0).any():
    raise ValueError("Invalid index_date/analysis_end_date for survival analysis")

raw_window_columns = [
    c for c in df.columns if any(c.endswith(f"_{tp}") for tp in time_points)
]
drop_columns = (
    raw_window_columns +
    [
        "index_date", "followup_end_date", "aitd_event_date",
        "aitd_event_window", "feature_cutoff_date",
        "last_complete_observation_window", "lab_feature_cutoff",
        "analysis_end_date", "early_censor_flag", "aitd_flag_complete_case",
        "rx_snapshot_landmark", "dx_snapshot_landmark", "cci_snapshot_landmark",
        "observed_followup_days",
        "m1_complete_flag", "m2_complete_flag", "m3_complete_flag",
        "m4_6_complete_flag", "m7_9_complete_flag", "m10_12_complete_flag",
        "m13_15_complete_flag", "m16_18_complete_flag", "m19_21_complete_flag",
        *timepoint_cols,
    ]
)

df = df.drop(columns=drop_columns, errors="ignore")

In [ ]:
df.columns.tolist()

In [ ]:
# Convert columns to numeric

df = df.apply(lambda col: pd.to_numeric(col, errors="coerce"))

In [ ]:
# Check missingness rate

missing_rate = df.isnull().mean() * 100
for col, rate in missing_rate.items():
    print(f"{col}: {rate:.2f}%")

In [ ]:
# Do not fit MICE here. It is fitted independently inside every training/CV
# fold by FoldwisePreprocessor below, preventing test/validation leakage.

In [ ]:
# Binary/remark missing values are intentionally left missing here and are
# filled inside each fold. Their clinical missing-data semantics require a
# separate analysis decision (missing indicator/category vs zero).

In [ ]:
# Retain the recorded SQL BMI. Invalid height/weight-associated BMI values
# were set to missing and will be imputed within training folds.
df.loc[df["bmi"] <= 0, "bmi"] = np.nan

In [ ]:
# Check missingness rate

missing_rate = df.isnull().mean() * 100
for col, rate in missing_rate.items():
    print(f"{col}: {rate:.2f}%")

In [ ]:
# SQL already calculated the four paper-defined annualized slopes using
# actual measurement dates and the feature cutoff.
required_slope_cols = ["ast_slope", "alt_slope", "scr_slope", "cholesterol_slope"]
missing_slopes = sorted(set(required_slope_cols) - set(df.columns))
if missing_slopes:
    raise KeyError(f"Run the updated feature3_extraction.sql; missing slopes: {missing_slopes}")

# Reporting snapshot before MICE, scaling, binary/remark filling, or slope filling.
# Table 1 and Table 2 must describe observed data, not imputed values.
report_features_before_imputation = df.copy()
report_features_before_imputation.to_csv(
    "report_features_before_imputation.csv", index=False, encoding="utf-8-sig"
)

In [ ]:
# No Python window-based slope is calculated. This avoids LOCF-created
# pseudo-observations and preserves the paper's annualized day-based scale.

In [ ]:
# Slopes are already present in df from SQL.

In [ ]:
# Check missingness rate

missing_rate = df.isnull().mean() * 100
for col, rate in missing_rate.items():
    print(f"{col}: {rate:.2f}%")

In [ ]:
# Fill missing slope values with zero

slope_cols = ["ast_slope", "alt_slope", "scr_slope", "cholesterol_slope"]
df[slope_cols] = df[slope_cols].fillna(0)

In [ ]:
# Check missingness rate

missing_rate = df.isnull().mean() * 100
for col, rate in missing_rate.items():
    print(f"{col}: {rate:.2f}%")

In [ ]:
df.to_csv("df_preprocessed.csv", index=False, encoding="utf-8-sig")
survival_df.to_csv("survival_metadata.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Fold-wise MICE and RobustScaler. This transformer is fitted only on the
# training portion supplied by train/test split or cross-validation.
from sklearn.base import BaseEstimator, TransformerMixin

class FoldwisePreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, max_iter=10, random_state=42):
        self.max_iter = max_iter
        self.random_state = random_state

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("FoldwisePreprocessor requires a pandas DataFrame")
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.binary_cols_ = [c for c in X.columns if c.endswith("_flag") or c.endswith("_rem")]
        self.continuous_cols_ = [c for c in X.columns if c not in self.binary_cols_]

        if self.continuous_cols_:
            self.imputer_ = IterativeImputer(
                max_iter=self.max_iter, random_state=self.random_state,
                keep_empty_features=True,
            )
            continuous_imputed = self.imputer_.fit_transform(X[self.continuous_cols_])
            self.scaler_ = RobustScaler().fit(continuous_imputed)
        return self

    def transform(self, X):
        if list(X.columns) != list(self.feature_names_in_):
            raise ValueError("Feature columns/order differ from the fitted training data")
        transformed = pd.DataFrame(index=X.index, columns=X.columns, dtype=float)
        if self.continuous_cols_:
            continuous_imputed = self.imputer_.transform(X[self.continuous_cols_])
            transformed.loc[:, self.continuous_cols_] = self.scaler_.transform(continuous_imputed)
        if self.binary_cols_:
            transformed.loc[:, self.binary_cols_] = X[self.binary_cols_].fillna(0).astype(float)
        return transformed

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_

In [ ]:
print("Global MICE/scaling disabled; preprocessing will be fitted fold-wise.")

## Model

In [ ]:
import os
import time
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from collections import OrderedDict

# Scikit-learn Ecosystem
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold,
    StratifiedShuffleSplit, ParameterGrid, cross_validate, cross_val_predict
)
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.multiclass import OneVsRestClassifier

# Model Zoo
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

# Performance Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve, make_scorer
)

# Imbalanced-learn (Critical for correct cross-validation with sampling)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.under_sampling import EditedNearestNeighbours
from imblearn.metrics import geometric_mean_score

# Explainability & Survival Analysis
import shap
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

In [ ]:
# Closed-network analysis: do not install packages from notebook cells.
# Install and validate requirements once during the approved internet window.
import sklearn, xgboost, imblearn, shap, lifelines
print({
    'scikit-learn': sklearn.__version__,
    'xgboost': xgboost.__version__,
    'imbalanced-learn': imblearn.__version__,
    'shap': shap.__version__,
    'lifelines': lifelines.__version__,
})

In [ ]:
RANDOM_STATE = 42
# Default to CPU for portability in the hospital closed network.
# Set XGB_DEVICE=cuda only after confirming an approved CUDA-capable setup.
XGB_DEVICE = os.environ.get('XGB_DEVICE', 'cpu').strip().lower()
if XGB_DEVICE not in {'cpu', 'cuda'}:
    raise ValueError("XGB_DEVICE must be 'cpu' or 'cuda'")
MODEL_N_JOBS = int(os.environ.get('MODEL_N_JOBS', '1'))
if MODEL_N_JOBS < 1:
    raise ValueError('MODEL_N_JOBS must be at least 1')
print(f'XGBoost device={XGB_DEVICE}, CV parallel jobs={MODEL_N_JOBS}')

In [ ]:
target_col = 'aitd_flag'
df_model = pd.read_csv("df_preprocessed.csv")

In [ ]:
person_ids = df_model['person_id'].copy()
df_model = df_model.drop(columns=['person_id'])

In [ ]:
X = df_model.drop(columns=[target_col])
y = df_model[target_col].astype(int)

In [ ]:
len(X.columns.tolist())

In [ ]:
lr_params = [
    {
        'clf__estimator__solver': ['lbfgs', 'newton-cg', 'sag'],
        'clf__estimator__penalty': ['l2', None],
        'clf__estimator__C': np.logspace(-3, 3, 7)
    },
    {
        'clf__estimator__solver': ['liblinear'],
        'clf__estimator__penalty': ['l1', 'l2'],
        'clf__estimator__C': np.logspace(-3, 3, 7)
    },
    {
        'clf__estimator__solver': ['saga'],
        'clf__estimator__penalty': ['l1', 'l2', 'elasticnet', None],
        'clf__estimator__C': np.logspace(-3, 3, 7),
        'clf__estimator__l1_ratio': [0, 0.5, 1]
    }
]
knn_params = {
    'clf__n_neighbors': np.arange(1, 31),
    'clf__weights': ['uniform', 'distance'],
    'clf__metric': ['euclidean', 'manhattan']
}
xgb_params = {
    'clf__max_depth': [3, 5, 8, 10, 12, 15, 18],
    'clf__min_child_weight': [0.1, 0.2, 0.3, 0.4],
    'clf__gamma': [0, 0.1, 0.2, 0.4, 0.5, 0.6, 0.7],
    'clf__subsample': [i/10.0 for i in range(6, 10)],
    'clf__colsample_bytree': [i/10.0 for i in range(6, 10)],
    'clf__n_estimators': [50, 100, 200, 500, 1000, 1500],
    'clf__learning_rate': [0.001, 0.01, 0.1, 0.2, 0.3]
}
ada_params = {
    'clf__n_estimators': [10, 50, 250, 1000, 1500, 2000, 2500, 3000],
    'clf__learning_rate': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
}

from imblearn.pipeline import Pipeline

models_and_params = {
    "KNN": (
        Pipeline([("preprocess", FoldwisePreprocessor(random_state=RANDOM_STATE)), ("clf", KNeighborsClassifier())]),
        [knn_params]
    ),
    "Logistic Regression": (
        Pipeline([("preprocess", FoldwisePreprocessor(random_state=RANDOM_STATE)), ("clf", OneVsRestClassifier(LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)))]),
        lr_params
    ),
    "XGBoost": (
        Pipeline([("preprocess", FoldwisePreprocessor(random_state=RANDOM_STATE)), ("clf", XGBClassifier(tree_method='hist', device=XGB_DEVICE, n_jobs=1, eval_metric='logloss', random_state=RANDOM_STATE))]),
        [xgb_params]
    ),
    "AdaBoost": (
        Pipeline([("preprocess", FoldwisePreprocessor(random_state=RANDOM_STATE)), ("clf", AdaBoostClassifier(random_state=RANDOM_STATE))]),
        [ada_params]
    )
}

In [ ]:
def evaluate_metrics(y_true, y_pred, y_proba):
    recall = recall_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    gmean = np.sqrt(recall * specificity)
    try:
        auroc = roc_auc_score(y_true, y_proba)
        auprc = average_precision_score(y_true, y_proba)
    except Exception:
        auroc, auprc = 0.0, 0.0

    return recall, precision, f1, gmean, acc, auroc, auprc

In [ ]:
# Data Splitting

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

split_manifest = pd.DataFrame({
    "person_id": person_ids.astype(int),
    "aitd_flag": y.astype(int),
    "split": np.where(X.index.isin(X_te.index), "holdout_test", "development"),
})
split_manifest.to_csv("data_split_manifest.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Preliminary default-model performance from development-only OOF predictions.
# The held-out test set is not accessed in this step.

quick_results = []
quick_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"⚡ Starting preliminary development 5-fold CV\n")

for name, (pipeline, _) in models_and_params.items():
    print(f"🚀 Training {name} ...", end=" ", flush=True)
    t0 = time.time()

    model = pipeline
    probas = cross_val_predict(
        model, X_tr, y_tr, cv=quick_cv, method="predict_proba", n_jobs=1
    )
    y_proba = probas[:, 1] if probas.shape[1] == 2 else probas.max(axis=1)
    y_pred = (y_proba >= 0.5).astype(int)

    metrics = evaluate_metrics(y_tr, y_pred, y_proba)
    recall, precision, f1, gmean, acc, auroc, auprc = metrics

    elapsed_time = time.time() - t0
    print(f"Done! ({elapsed_time:.2f}s)")

    result = OrderedDict({
        "Model": name,
        "Recall": round(recall, 4),
        "Precision": round(precision, 4),
        "F1-score": round(f1, 4),
        "G-mean": round(gmean, 4),
        "Accuracy": round(acc, 4),
        "AUROC": round(auroc, 4),
        "AUPRC": round(auprc, 4),
        "Time (s)": round(elapsed_time, 2)
    })
    quick_results.append(result)

quick_df = (
    pd.DataFrame(quick_results)
    .sort_values(by="F1-score", ascending=False)
    .reset_index(drop=True)
)
quick_df.to_csv("quick_development_cv_performance.csv", index=False, encoding="utf-8-sig")

print("\n" + "="*80)
print("📊 Development OOF Performance Summary (Sorted by F1-score)")
print("="*80)
print(quick_df.to_string(index=False))

In [ ]:
# Configuration for recursive feature elimination on the development split only.
# The held-out test split is never used for feature selection.
RFECV_MODELS = ["AdaBoost", "XGBoost"]
RFECV_SCORING = "accuracy"
MIN_FEATURES_TO_SELECT = 1
rfecv_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
# Custom recursive feature elimination with stratified 5-fold validation.
# At every step, fold-level feature importances are averaged and exactly one
# least-important feature is removed. Importances are used but not persisted.
from sklearn.base import clone

if not hasattr(X_tr, "columns"):
    raise TypeError("X_tr must be a pandas DataFrame with feature names")
feature_names = X_tr.columns.tolist()

def recursive_elimination_cv(estimator, X_data, y_data, cv, min_features=1):
    initial_features = list(X_data.columns)
    current_features = initial_features.copy()
    elimination_order = []
    history = []

    while len(current_features) >= min_features:
        fold_scores = []
        fold_importances = []

        for train_idx, valid_idx in cv.split(X_data, y_data):
            fold_estimator = clone(estimator)
            if "n_jobs" in fold_estimator.get_params(deep=False):
                fold_estimator.set_params(n_jobs=1)

            X_fold_train_raw = X_data.iloc[train_idx].loc[:, current_features]
            X_fold_valid_raw = X_data.iloc[valid_idx].loc[:, current_features]
            y_fold_train = y_data.iloc[train_idx]
            y_fold_valid = y_data.iloc[valid_idx]

            fold_preprocessor = FoldwisePreprocessor(random_state=RANDOM_STATE)
            X_fold_train = fold_preprocessor.fit_transform(X_fold_train_raw, y_fold_train)
            X_fold_valid = fold_preprocessor.transform(X_fold_valid_raw)

            fold_estimator.fit(X_fold_train, y_fold_train)
            fold_pred = fold_estimator.predict(X_fold_valid)
            fold_scores.append(accuracy_score(y_fold_valid, fold_pred))

            if not hasattr(fold_estimator, "feature_importances_"):
                raise AttributeError(f"{type(fold_estimator).__name__} has no feature_importances_")
            importance = np.asarray(fold_estimator.feature_importances_, dtype=float)
            if importance.shape[0] != len(current_features) or not np.all(np.isfinite(importance)):
                raise ValueError("Invalid feature importance vector")
            fold_importances.append(importance)

        mean_importance = np.mean(np.vstack(fold_importances), axis=0)
        mean_accuracy = float(np.mean(fold_scores))
        std_accuracy = float(np.std(fold_scores))

        removed_feature = None
        if len(current_features) > min_features:
            minimum = float(np.min(mean_importance))
            # Deterministic tie handling: remove the first tied feature in column order.
            tied = np.flatnonzero(np.isclose(mean_importance, minimum, rtol=1e-12, atol=1e-15))
            remove_idx = int(tied[0])
            removed_feature = current_features[remove_idx]

        history.append({
            "n_features": len(current_features),
            "mean_cv_accuracy": mean_accuracy,
            "std_cv_accuracy": std_accuracy,
            "removed_feature": removed_feature,
            "features": tuple(current_features),
        })
        print(
            f"n={len(current_features):>3} | accuracy={mean_accuracy:.4f} "
            f"(+/- {std_accuracy:.4f}) | remove={removed_feature}"
        )

        if removed_feature is None:
            break
        elimination_order.append(removed_feature)
        current_features.pop(remove_idx)

    history_df = pd.DataFrame(history)
    best_accuracy = float(history_df["mean_cv_accuracy"].max())
    best_rows = history_df[
        np.isclose(history_df["mean_cv_accuracy"], best_accuracy, rtol=1e-12, atol=1e-15)
    ]
    # If accuracy ties exactly, choose the smaller feature set.
    best_row = best_rows.sort_values("n_features", ascending=True).iloc[0]
    selected = list(best_row["features"])

    removed_before_best = elimination_order[:len(initial_features) - len(selected)]
    ranking = {feature: 1 for feature in selected}
    for rank, feature in enumerate(reversed(removed_before_best), start=2):
        ranking[feature] = rank

    return {
        "selected_features": selected,
        "n_features": len(selected),
        "best_cv_score": best_accuracy,
        "history": history_df,
        "ranking": ranking,
    }

print(f"Total original features: {X_tr.shape[1]}")
selected_models = {name: models_and_params[name][0] for name in RFECV_MODELS}
rfecv_results = {}

fig, axes = plt.subplots(1, len(selected_models), figsize=(7 * len(selected_models), 5))
if len(selected_models) == 1:
    axes = [axes]

for ax, (name, pipeline) in zip(axes, selected_models.items()):
    print(f"\n===== {name}: recursive 5-fold feature elimination =====")
    estimator = pipeline.named_steps["clf"]
    result = recursive_elimination_cv(
        estimator, X_tr, y_tr, rfecv_cv, min_features=MIN_FEATURES_TO_SELECT
    )
    rfecv_results[name] = result

    plot_df = result["history"].sort_values("n_features")
    ax.plot(plot_df["n_features"], plot_df["mean_cv_accuracy"], marker="o")
    ax.fill_between(
        plot_df["n_features"],
        plot_df["mean_cv_accuracy"] - plot_df["std_cv_accuracy"],
        plot_df["mean_cv_accuracy"] + plot_df["std_cv_accuracy"],
        alpha=0.2,
    )
    ax.axvline(
        result["n_features"], linestyle="--", color="black",
        label=f"n={result['n_features']}\naccuracy={result['best_cv_score']:.3f}",
    )
    ax.set_title(f"Recursive 5-fold accuracy ({name})")
    ax.set_xlabel("Number of remaining features")
    ax.set_ylabel("Mean validation accuracy")
    ax.legend()

    history_export = result["history"].copy()
    history_export["features"] = history_export["features"].apply(lambda x: "|".join(x))
    history_export.to_csv(
        f"recursive_accuracy_history_{name.lower()}.csv",
        index=False, encoding="utf-8-sig",
    )
    print(f"Best feature count: {result['n_features']}")
    print(f"Best mean validation accuracy: {result['best_cv_score']:.4f}")
    print(result["selected_features"])

plt.tight_layout()
plt.show()

In [ ]:
# Save each model's best-accuracy feature set and use their intersection.
manifest_frames = []
for model_name, result in rfecv_results.items():
    model_manifest = pd.DataFrame({
        "selector_model": model_name,
        "scoring": RFECV_SCORING,
        "best_cv_score": result["best_cv_score"],
        "feature": feature_names,
        "selected": [f in set(result["selected_features"]) for f in feature_names],
        "ranking": [result["ranking"].get(f, np.nan) for f in feature_names],
    }).sort_values(["selected", "ranking", "feature"], ascending=[False, True, True])
    model_manifest.to_csv(
        f"rfecv_feature_selection_{model_name.lower()}.csv",
        index=False, encoding="utf-8-sig"
    )
    manifest_frames.append(model_manifest)

rfecv_selection_manifest = pd.concat(manifest_frames, ignore_index=True)
rfecv_selection_manifest.to_csv(
    "rfecv_feature_selection_all_models.csv", index=False, encoding="utf-8-sig"
)

ada_set = set(rfecv_results["AdaBoost"]["selected_features"])
xgb_set = set(rfecv_results["XGBoost"]["selected_features"])
common_set = ada_set & xgb_set
selected_features = [feature for feature in feature_names if feature in common_set]
if not selected_features:
    raise RuntimeError("AdaBoost and XGBoost selected-feature intersection is empty")

missing_selected = sorted(set(selected_features) - set(X_tr.columns))
if missing_selected:
    raise KeyError(f"Selected features missing from the model matrix: {missing_selected}")

X_tr_sel = X_tr.loc[:, selected_features].copy()
X_te_sel = X_te.loc[:, selected_features].copy()
y_tr_sel = y_tr.copy()
y_te_sel = y_te.copy()
X_sel = X.loc[:, selected_features].copy()

FEATURE_SET_NAME = "adaboost_xgboost_intersection"
pd.DataFrame({"feature": selected_features}).to_csv(
    "recursive_selected_feature_intersection.csv", index=False, encoding="utf-8-sig"
)

print(f"AdaBoost selected: {len(ada_set)}")
print(f"XGBoost selected: {len(xgb_set)}")
print(f"Shared/final features: {len(selected_features)}")
print("Using the AdaBoost-XGBoost intersection for downstream modeling")
print(selected_features)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import hashlib

scorer = make_scorer(f1_score, average='macro')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

feature_signature = hashlib.sha256("\n".join(selected_features).encode("utf-8")).hexdigest()[:12]
RESULTS_FILE = f"tuning_results_log_{FEATURE_SET_NAME}_{feature_signature}.csv"
samplers = ["none", "b-smote", "enn", "b-smote-enn"]

def build_final_pipeline(model_pipeline, sampler_name):
    preprocess_step = clone(model_pipeline.named_steps['preprocess'])
    model_step = clone(model_pipeline.named_steps['clf'])
    steps = [("preprocess", preprocess_step)]
    if sampler_name == "b-smote":
        steps.append(("bsmote", BorderlineSMOTE(random_state=RANDOM_STATE)))
    elif sampler_name == "enn":
        steps.append(("enn", EditedNearestNeighbours()))
    elif sampler_name == "b-smote-enn":
        steps.append(("bsmote", BorderlineSMOTE(random_state=RANDOM_STATE)))
        steps.append(("enn", EditedNearestNeighbours()))

    steps.append(("clf", model_step))
    return ImbPipeline(steps)

if os.path.exists(RESULTS_FILE):
    final_df = pd.read_csv(RESULTS_FILE)
    print(f"🔄 Existing log found. Resuming from previous state...")
else:
    final_df = pd.DataFrame()

total_combos = len(models_and_params) * len(samplers)
current_idx = 0

for model_name, (pipeline_obj, param_grid) in models_and_params.items():
    for sampler_name in samplers:
        current_idx += 1

        if not final_df.empty:
            is_done = final_df[(final_df['Model'] == model_name) & (final_df['Sampler'] == sampler_name)]
            if not is_done.empty:
                print(f"⏩ [{current_idx}/{total_combos}] Skipping: {model_name} + {sampler_name}")
                continue

        print(f"🚀 [{current_idx}/{total_combos}] Tuning: {model_name} | Sampler: {sampler_name}...")
        start_time = datetime.now()

        current_pipeline = build_final_pipeline(pipeline_obj, sampler_name)

        # grid = GridSearchCV(
        #     estimator=current_pipeline,
        #     param_grid=param_grid,
        #     scoring=scorer,
        #     cv=cv,
        #     n_jobs=-1,
        #     verbose=2
        # )
        grid = RandomizedSearchCV(
            estimator=current_pipeline,
            param_distributions=param_grid,
            n_iter=50,
            scoring=scorer,
            cv=cv,
            n_jobs=MODEL_N_JOBS,
            verbose=3,
            random_state=RANDOM_STATE
        )

        try:
            grid.fit(X_tr_sel, y_tr_sel)

            best_params_native = {
                key: (value.item() if isinstance(value, np.generic) else value)
                for key, value in grid.best_params_.items()
            }
            res = {
                "Model": model_name,
                "Sampler": sampler_name,
                "Best_Params": repr(best_params_native),
                "Best_CV_Macro_F1": float(grid.best_score_),
                "Timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            }

            res_df = pd.DataFrame([res])
            res_df.to_csv(RESULTS_FILE, mode='a', index=False, header=not os.path.exists(RESULTS_FILE))

            final_df = pd.concat([final_df, res_df], ignore_index=True)

            print(f"✅ Success: {model_name}({sampler_name}) -> development CV macro-F1: {grid.best_score_:.4f}")

        except Exception as e:
            print(f"❌ Error in {model_name} + {sampler_name}: {e}")

print("\n" + "="*50)
print("🎯 Hyperparameter Tuning Completed Successfully!")
print("="*50)

In [ ]:
# Detailed Cross-Validation Performance Report for Best Hyperparameters across 16 Model-Sampler Combinations

if not os.path.exists(RESULTS_FILE):
    print(f"❌ Error: {RESULTS_FILE} not found. Please complete the tuning process first.")
else:
    results_df = pd.read_csv(RESULTS_FILE)

    scoring_metrics = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'roc_auc': 'roc_auc'
    }

    print(f"✅ Starting detailed CV validation for {len(results_df)} model combinations.\n")
    detailed_cv_rows = []

    for idx, row in results_df.iterrows():
        model_name = row['Model']
        sampler_name = row['Sampler']
        print(row['Best_Params'])
        best_params = ast.literal_eval(row['Best_Params'])

        print("="*90)
        print(f" REPORT #{idx+1}: {model_name} with {sampler_name} ".center(90, "="))
        print("="*90)

        base_pipeline = models_and_params[model_name][0]

        final_pipe = build_final_pipeline(base_pipeline, sampler_name)
        final_pipe.set_params(**best_params)

        cv_results = cross_validate(
            final_pipe, X_tr_sel, y_tr_sel,
            cv=cv,
            scoring=scoring_metrics,
            return_train_score=False
        )

        summary_data = []
        for metric_name in scoring_metrics.keys():
            scores = cv_results[f'test_{metric_name}']
            mean_score = np.mean(scores)
            std_score = np.std(scores)

            row_data = {
                "METRIC": metric_name.upper(),
                "FOLD 1": scores[0],
                "FOLD 2": scores[1],
                "FOLD 3": scores[2],
                "FOLD 4": scores[3],
                "FOLD 5": scores[4],
                "MEAN": mean_score,
                "STD": std_score
            }
            summary_data.append(row_data)
            detailed_cv_rows.append({"Model": model_name, "Sampler": sampler_name, **row_data})

        display_df = pd.DataFrame(summary_data)
        pd.options.display.float_format = '{:.4f}'.format
        print(display_df.to_string(index=False))
        print("\n" + "-"*90 + "\n")

    pd.DataFrame(detailed_cv_rows).to_csv(
        "development_5fold_cv_performance.csv", index=False, encoding="utf-8-sig"
    )

In [ ]:
# Final evaluation on the test set for all 16 combinations, grouped by model name

import ast

final_test_results = []

print(f"🔍 Evaluating all {len(results_df)} combinations on the Test Set (Grouped by Model)...")

for idx, row in results_df.iterrows():
    m_name = row['Model']
    s_name = row['Sampler']
    params = ast.literal_eval(row['Best_Params'])
    base_pipe = models_and_params[m_name][0]
    pipe = build_final_pipeline(base_pipe, s_name)
    pipe.set_params(**params)
    pipe.fit(X_tr_sel, y_tr_sel)
    y_p = pipe.predict(X_te_sel)
    y_prob = pipe.predict_proba(X_te_sel)[:, 1] if hasattr(pipe, "predict_proba") else np.zeros(len(y_p))

    m = evaluate_metrics(y_te_sel, y_p, y_prob)

    final_test_results.append({
        "Model": m_name,
        "Sampler": s_name,
        "F1": m[2],
        "Recall": m[0],
        "Precision": m[1],
        "G-mean": m[3],
        "Accuracy": m[4],
        "AUROC": m[5],
        "AUPRC": m[6]
    })

test_report_df = pd.DataFrame(final_test_results)

test_report_df = test_report_df.sort_values(by=["Model", "Sampler"])
test_report_df.to_csv("final_holdout_performance.csv", index=False, encoding="utf-8-sig")

print("\n" + "="*95)
print(" 📋 FINAL TEST SET PERFORMANCE REPORT (GROUPED BY MODEL) ".center(95, "="))
print("="*95)
pd.options.display.float_format = '{:.4f}'.format
print(test_report_df.to_string(index=False))
print("="*95)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. 스타일 및 테마 설정
sns.set_theme(style="whitegrid")
custom_colors = ["#F4C0BD", "#C3D792", "#91DEE0", "#E1C3F7"]

def draw_performance_plots(df, metrics, model_order, sampler_order):
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    axes = axes.flatten()

    legend_map = dict(zip(sampler_order, ['Raw', 'B-SMT', 'ENN', 'B-SMT ENN']))
    custom_colors = ["#F4C0BD", "#C3D792", "#91DEE0", "#E1C3F7"]

    for i, metric in enumerate(metrics):
        ax = axes[i]

        sns.barplot(
            data=df,
            x="Model",
            y=metric,
            hue="Sampler",
            order=model_order,
            hue_order=sampler_order,
            ax=ax,
            palette=custom_colors,
            edgecolor=None,
            linewidth=0
        )

        # 막대 사이 간격 추가 및 텍스트 주석 처리
        width_scale = 0.8
        for p in ax.patches:
            # 1. 막대 너비 및 위치 조정 (간격 생성)
            current_width = p.get_width()
            if current_width > 0: # 유효한 패치만 수정
                new_width = current_width * width_scale
                diff = current_width - new_width
                p.set_width(new_width)
                p.set_x(p.get_x() + diff / 2)

                # 2. 값 표시 (0이어도 표시되도록 조건 제거)
                val = p.get_height()
                # nan 값이 아니라면 0 포함 모두 출력
                if not np.isnan(val):
                    ax.annotate(f'{val:.3f}',
                                 (p.get_x() + p.get_width() / 2., val),
                                 ha='center', va='bottom',
                                 fontsize=9, fontweight='bold',
                                 xytext=(0, 3),
                                 textcoords='offset points')

        ax.set_title(f"{metric}", fontsize=15, fontweight='bold', pad=15)
        ax.set_ylim(0, 1.15)
        ax.set_xlabel("")
        ax.set_ylabel(metric, fontsize=12)

        handles, _ = ax.get_legend_handles_labels()
        ax.legend(handles, legend_map.values(), title="Strategy",
                  loc='upper right', frameon=True, fontsize=10)

    plt.suptitle("Model Performance Analysis by Resampling Strategy",
                 fontsize=22, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

metrics = ["AUPRC", "F1", "G-mean", "Recall"]
model_order = ['XGBoost', 'AdaBoost', 'KNN', 'Logistic Regression']
sampler_order = ['none', 'b-smote', 'enn', 'b-smote-enn']

# test_report_df 데이터프레임이 준비된 상태에서 실행
draw_performance_plots(test_report_df, metrics, model_order, sampler_order)

In [ ]:
# Train the leakage-safe XGBoost + B-SMOTE pipeline and explain the held-out test set.

import shap
import ast
import matplotlib.pyplot as plt

if os.path.exists(RESULTS_FILE):
    results_df = pd.read_csv(RESULTS_FILE)

    target_row = results_df[(results_df['Model'] == 'XGBoost') &
                            (results_df['Sampler'] == 'b-smote')].iloc[0]

    best_params = ast.literal_eval(target_row['Best_Params'])
    print(f"✅ Loaded Best Params for XGBoost (b-smote): {best_params}")

    final_xgb_pipe = build_final_pipeline(models_and_params['XGBoost'][0], 'b-smote')
    final_xgb_pipe.set_params(**best_params)
    final_xgb_pipe.fit(X_tr_sel, y_tr_sel)

    X_te_explain = final_xgb_pipe.named_steps['preprocess'].transform(X_te_sel)
    final_xgb = final_xgb_pipe.named_steps['clf']
    explainer = shap.TreeExplainer(final_xgb)
    shap_values = explainer.shap_values(X_te_explain)
    if isinstance(shap_values, list):
        shap_values_array = np.asarray(shap_values[-1], dtype=float)
    else:
        shap_values_array = np.asarray(shap_values, dtype=float)
    if shap_values_array.ndim == 3:
        shap_values_array = shap_values_array[:, :, -1]
    if shap_values_array.shape != X_te_explain.shape:
        raise ValueError("SHAP matrix is not aligned with held-out feature values")

    shap_feature_values = X_te_explain.copy()
    shap_feature_values.insert(0, "person_id", person_ids.loc[X_te_sel.index].astype(int).to_numpy())
    shap_feature_values.to_csv("shap_feature_values_heldout.csv", index=False, encoding="utf-8-sig")
    shap_values_df = pd.DataFrame(shap_values_array, columns=X_te_explain.columns)
    shap_values_df.insert(0, "person_id", person_ids.loc[X_te_sel.index].astype(int).to_numpy())
    shap_values_df.to_csv("shap_values_heldout.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame({
        "feature": X_te_explain.columns,
        "mean_abs_shap": np.abs(shap_values_array).mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False).to_csv(
        "shap_importance.csv", index=False, encoding="utf-8-sig"
    )

    print("\n📊 Generating SHAP Summary Plot...")
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values_array, X_te_explain, plot_type="bar")

    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values_array, X_te_explain)
    plt.show()

else:
    print("❌ Results file not found.")

In [ ]:
# Select thresholds from development-set out-of-fold predictions, freeze
# them, and evaluate them once on the held-out test set.
if os.path.exists(RESULTS_FILE):
    results_df = pd.read_csv(RESULTS_FILE)
    target_row = results_df[(results_df['Model'] == 'XGBoost') &
                            (results_df['Sampler'] == 'b-smote')].iloc[0]
    best_params = ast.literal_eval(target_row['Best_Params'])
    print(f"✅ Loaded Best Params for XGBoost (b-smote): {best_params}")

    threshold_pipe = build_final_pipeline(models_and_params['XGBoost'][0], 'b-smote')
    threshold_pipe.set_params(**best_params)
    oof_prob = cross_val_predict(
        threshold_pipe, X_tr_sel, y_tr_sel, cv=cv,
        method='predict_proba', n_jobs=1,  # keep CUDA fits serial
    )[:, 1]

    oof_precision, oof_recall, oof_thresholds = precision_recall_curve(y_tr_sel, oof_prob)
    if len(oof_thresholds) == 0:
        raise RuntimeError("No threshold candidates from OOF predictions")
    oof_f1 = [f1_score(y_tr_sel, (oof_prob >= t).astype(int)) for t in oof_thresholds]

    def top_percent_threshold(y_prob, percent):
        return float(np.percentile(y_prob, 100 - percent))

    thresholds_dict = {
        "A (Top 1%)": top_percent_threshold(oof_prob, 1),
        "B (Top 5%)": top_percent_threshold(oof_prob, 5),
        "C (Optimal F1)": float(oof_thresholds[int(np.argmax(oof_f1))]),
        "D (Top 15%)": top_percent_threshold(oof_prob, 15),
        "E (Top 25%)": top_percent_threshold(oof_prob, 25),
    }
    pd.DataFrame(
        [{"name": name, "threshold": threshold} for name, threshold in thresholds_dict.items()]
    ).to_csv("thresholds_selected_from_development_oof.csv", index=False, encoding="utf-8-sig")

    final_xgb_pipe = clone(threshold_pipe).fit(X_tr_sel, y_tr_sel)
    y_test_prob = final_xgb_pipe.predict_proba(X_te_sel)[:, 1]
    heldout_predictions = pd.DataFrame({
        "person_id": person_ids.loc[X_te_sel.index].astype(int).to_numpy(),
        "y_true": y_te_sel.astype(int).to_numpy(),
        "y_probability": y_test_prob,
    })
    heldout_predictions.to_csv(
        "heldout_predictions_xgboost_bsmote.csv", index=False, encoding="utf-8-sig"
    )
    test_precision, test_recall, _ = precision_recall_curve(y_te_sel, y_test_prob)
    test_ap = average_precision_score(y_te_sel, y_test_prob)

    plt.figure(figsize=(10, 8))
    plt.plot(test_recall, test_precision, color='navy', lw=2, label=f"Held-out PR Curve (AP={test_ap:.4f})")
    plt.fill_between(test_recall, test_precision, alpha=0.2, color='navy')

    print("\n--- Frozen OOF Threshold Performance on Held-out Test ---")
    print(f"{'Name':<15} | {'Thr':<8} | {'Test %':<8} | {'Precision':<10} | {'Recall':<10}")
    print("-" * 68)
    for name, thr in thresholds_dict.items():
        y_pred_final = (y_test_prob >= thr).astype(int)
        p = precision_score(y_te_sel, y_pred_final, zero_division=0)
        r = recall_score(y_te_sel, y_pred_final, zero_division=0)
        actual_test_percent = round(y_pred_final.mean() * 100, 1)
        plt.scatter(r, p, s=150, edgecolors='black', zorder=5,
                    label=f"{name}: Thr={thr:.3f} (Test {actual_test_percent}%)")
        print(f"{name:<15} | {thr:.4f} | {actual_test_percent:>7}% | {p:.4f} | {r:.4f}")

    plt.xlabel("Recall (Sensitivity)")
    plt.ylabel("Precision (PPV)")
    plt.title("Frozen Development-OOF Thresholds: XGBoost + Borderline-SMOTE")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("❌ Results file not found.")

In [ ]:
# Kaplan-Meier analysis on the untouched test patients. Survival time is
# event/censoring date minus index date; frozen development OOF thresholds
# from the previous cell are reused without test-set optimization.
survival_meta = pd.read_csv("survival_metadata.csv")
survival_meta = survival_meta.set_index("person_id")
test_person_ids = person_ids.loc[X_te_sel.index].astype(int)
if survival_meta.index.duplicated().any():
    raise ValueError("Duplicate person_id in survival_metadata.csv")
test_survival = survival_meta.loc[test_person_ids.to_numpy()].copy()
time_test = test_survival["followup_time_days"].to_numpy(dtype=float)
event_test = y_te_sel.to_numpy(dtype=int)
if len(time_test) != len(y_test_prob) or not np.array_equal(test_survival.index.to_numpy(), test_person_ids.to_numpy()):
    raise RuntimeError("Survival metadata is not aligned with the held-out test set")

def plot_km_analysis(time, event, prob, threshold, title):
    high_risk = prob >= threshold
    low_risk = prob < threshold

    if high_risk.sum() == 0 or low_risk.sum() == 0:
        print(f"Skipping {title}: one risk group is empty")
        return None

    kmf = KaplanMeierFitter()
    plt.figure(figsize=(7, 5))

    kmf.fit(time[low_risk], event[low_risk], label="Low Risk")
    kmf.plot_cumulative_density(color="green", linewidth=2)

    kmf.fit(time[high_risk], event[high_risk], label="High Risk")
    kmf.plot_cumulative_density(color="red", linewidth=2)

    lr_results = logrank_test(
        time[high_risk], time[low_risk],
        event_observed_A=event[high_risk], event_observed_B=event[low_risk],
    )

    plt.title(f"{title}\nLog-rank p-value: {lr_results.p_value:.4e}")
    plt.xlabel("Days")
    plt.ylabel("Cumulative Incidence of AITD")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()
    return {
        "Cutoff": title.replace("KM Analysis: ", ""),
        "Threshold": float(threshold),
        "N_high_risk": int(high_risk.sum()),
        "N_low_risk": int(low_risk.sum()),
        "Events_high_risk": int(event[high_risk].sum()),
        "Events_low_risk": int(event[low_risk].sum()),
        "Logrank_p_value": float(lr_results.p_value),
    }

km_summary_rows = []
for label, thr in thresholds_dict.items():
    km_row = plot_km_analysis(time_test, event_test, y_test_prob, thr, f"KM Analysis: {label}")
    if km_row is not None:
        km_summary_rows.append(km_row)
pd.DataFrame(km_summary_rows).to_csv(
    "km_threshold_summary.csv", index=False, encoding="utf-8-sig"
)

heldout_predictions = pd.DataFrame({
    "person_id": test_person_ids.to_numpy(),
    "y_true": event_test,
    "y_probability": y_test_prob,
    "followup_time_days": time_test,
})
heldout_predictions.to_csv(
    "heldout_predictions_xgboost_bsmote.csv", index=False, encoding="utf-8-sig"
)

def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "NPV": tn / (tn + fn) if (tn + fn) > 0 else 0
    }

summary_rows = []
for name, thr in thresholds_dict.items():
    actual_percent = (y_test_prob >= thr).mean() * 100
    m = compute_metrics(event_test, y_test_prob, thr)
    summary_rows.append({
        "Cutoff": name,
        "Pop_Share": f"{actual_percent:.1f}%",
        "Threshold": round(thr, 4),
        **{k: round(v, 3) for k, v in m.items()}
    })

df_final_metrics = pd.DataFrame(summary_rows)
df_final_metrics.to_csv("final_threshold_performance.csv", index=False, encoding="utf-8-sig")
print("\n" + "="*80)
print(" FINAL PERFORMANCE SUMMARY ACROSS THRESHOLDS ".center(80, "="))
print("="*80)
print(df_final_metrics.to_string(index=False))